# Executive Summary

## Project Objective

This experiment studies whether machine learning classifiers can predict the **daily return spreads** between **NVIDIA (NVDA)** and **Tesla (TSLA)** using a combination of:

1. **accounting fundamentals**, which capture slow-moving (quarterly) firm characteristics, and  
2. **momentum / technical indicators**, which capture short-run (daily) market dynamics.

The idea is straightforward: the model forecasts **which stock will outperform the other tomorrow**. That prediction is then converted into a **daily long-short trading strategy**.

---

## Prediction Target

Let $P_{i,t}$ denote the adjusted closing price of stock $i \in \{\text{NVDA}, \text{TSLA}\}$ on day $t$.  
Daily simple return is defined as:

$$
r_{i,t}=\frac{P_{i,t}}{P_{i,t-1}}-1
$$

The daily **spread return** between the two stocks is:

$$
s_t = r_{\text{NVDA},t} - r_{\text{TSLA},t}
$$

The binary classification target is the **sign of tomorrow’s spread**:

$$
y_t = \mathbf{1}\{s_{t+1} > 0\}
$$

So:

- $y_t = 1$: NVDA outperforms TSLA tomorrow  
- $y_t = 0$: TSLA outperforms NVDA tomorrow

This turns the problem into a **binary classification task**.

---

## Model

The baseline model is **logistic regression**. For feature vector $X_t$, the model estimates:

$$
p_t = \Pr(y_t = 1 \mid X_t)
$$

with logistic form:

$$
p_t = \frac{1}{1+\exp\left(-(\beta_0 + \beta'X_t)\right)}
$$

The classification rule is:

$$
\hat{y}_t =
\begin{cases}
1, & p_t \ge 0.5 \\
0, & p_t < 0.5
\end{cases}
$$

This means:

- if $p_t \ge 0.5$, the model predicts NVDA will beat TSLA tomorrow;
- otherwise, it predicts TSLA will beat NVDA tomorrow.

---

## Trading Rule

The trading rule is:

$$
\begin{aligned}
&d_t= \begin{cases}1, & p_t \geq \tau_L \\ 0, & \tau_S<p_t<\tau_L \\ -1, & p_t \leq \tau_S\end{cases}
\end{aligned}
$$

with:

$$\tau_L=0.8, \quad \tau_S=0.2$$

So:
- $d_t=1$ : long NVDA, short TSLA
- $d_t=-1$ : long TSLA, short NVDA
- $d_t=0$ : hold cash / no position

With equal dollar weights, daily long-short portfolio return is:

$$
R_{t+1}^{LS}
= \frac{1}{2}\hat{d}_t\, r_{\text{NVDA},t+1}
-\frac{1}{2}\hat{d}_t\, r_{\text{TSLA},t+1}
= \frac{1}{2}\hat{d}_t\, s_{t+1}
$$

This makes the trading rule directly tied to the predicted sign of the spread.

---

## Data and Sample Split

The raw dataset is a daily panel containing:

- adjusted prices,
- trading volume,
- quarterly accounting fundamentals.

The experiment only keeps the two stocks:

- **NVDA**
- **TSLA**

The sample is split chronologically to avoid look-ahead bias:

- **Training sample:** 2011-01-03 to 2021-12-31  
- **Testing sample:** 2022-01-03 to 2025-12-31

---

## Feature Design

A key design choice is that the target is a **pair spread**, so features are also constructed in **pair-difference form**.

For any stock-level variable $z_{i,t}$, the pair feature is:

$$
x_t^{(z)} = z_{\text{NVDA},t} - z_{\text{TSLA},t}
$$


The final model uses **15 pair-difference features**.

---

## Accounting Features

These variables are computed at the stock level and then transformed into NVDA-minus-TSLA differences.

### 1. Operating margin difference

$$
\text{op\_margin}_{i,t} = \frac{\text{oiadpq}_{i,t}}{\text{saleq}_{i,t}}
$$

Feature:

$$
\text{op\_margin\_diff}_t
=
\text{op\_margin}_{\text{NVDA},t}
-
\text{op\_margin}_{\text{TSLA},t}
$$

Interpretation: relative operating profitability.

---

### 2. Net margin difference

$$
\text{net\_margin}_{i,t} = \frac{\text{niq}_{i,t}}{\text{saleq}_{i,t}}
$$

Interpretation: relative bottom-line profitability.

---

### 3. Leverage difference

$$
\text{leverage}_{i,t} = \frac{\text{dlttq}_{i,t}+\text{dlcq}_{i,t}}{\text{atq}_{i,t}}
$$

Interpretation: relative debt burden.

---

### 4. Cash ratio difference

$$
\text{cash\_ratio}_{i,t} = \frac{\text{cheq}_{i,t}}{\text{atq}_{i,t}}
$$

Interpretation: relative balance-sheet liquidity.

---

### 5. R&D intensity difference

$$
\text{rnd\_intensity}_{i,t} = \frac{\text{xrdq}_{i,t}}{\text{saleq}_{i,t}}
$$

Interpretation: relative innovation intensity.

---

### 6. Capex intensity difference

$$
\text{capex\_intensity}_{i,t} = \frac{\text{capxy}_{i,t}}{\text{atq}_{i,t}}
$$

Interpretation: relative investment intensity.

---

### 7. Asset turnover difference

$$
\text{asset\_turnover}_{i,t} = \frac{\text{saleq}_{i,t}}{\text{atq}_{i,t}}
$$

Interpretation: relative operating efficiency.

---

### 8. Log market capitalization difference

$$
\text{log\_mcap}_{i,t} = \log\left(P_{i,t}\times \text{cshoq}_{i,t}\right)
$$

Interpretation: relative firm size.

---

## Momentum and Technical Features

These variables are designed to provide daily market timing signals.

---

### 9. 5-day momentum difference

$$
\text{mom5}_{i,t} = \frac{P_{i,t}}{P_{i,t-5}} - 1
$$

Interpretation: short-run trend.

---

### 10. 21-day momentum difference

$$
\text{mom21}_{i,t} = \frac{P_{i,t}}{P_{i,t-21}} - 1
$$

Interpretation: approximately one-month trend.

---

### 11. 63-day momentum difference

$$
\text{mom63}_{i,t} = \frac{P_{i,t}}{P_{i,t-63}} - 1
$$

Interpretation: medium-horizon trend.

---

### 12. 20-day moving-average gap difference

Let $MA_{20}(P_i)_t$ be the 20-day moving average of price.

$$
\text{ma\_gap20}_{i,t} = \frac{P_{i,t}}{MA_{20}(P_i)_t} - 1
$$

Interpretation: how stretched the stock is relative to its recent trend.

---

### 13. 14-day RSI difference

RSI is constructed from average gains and losses over a 14-day window.

Interpretation: relative overbought / oversold condition.

---

### 14. 21-day realized volatility difference

$$
\text{rv21}_{i,t} = \sqrt{252}\cdot \text{Std}(r_{i,t-20},\ldots,r_{i,t})
$$

Interpretation: relative recent risk.

---

### 15. 20-day volume shock difference

Let $\overline{V}_{20,i,t}$ be the 20-day moving average of volume:

$$
\text{vol\_shock20}_{i,t} = \log\left(\frac{\text{volume}_{i,t}}{\overline{V}_{20,i,t}}\right)
$$

Interpretation: unusual trading activity relative to normal volume.

---

## Important Implementation Choices

### Clipping extreme values

Feature values are clipped at the 1st and 99th percentiles using the **training sample only**.

---

### Standardization

All features are standardized using the training sample mean and standard deviation:

$$
x_{j,t}^{scaled}
=
\frac{x_{j,t} - \mu_j^{train}}{\sigma_j^{train}}
$$

---

### No leakage

All predictors are built using information available at time $t$ to make predictions on $t+1$.

---

## Classification Performance Metrics

### Accuracy

$$
\text{Accuracy} = \frac{1}{n}\sum_{t=1}^{n}\mathbf{1}\{\hat{y}_t = y_t\}
$$

---

### Balanced Accuracy

$$
\text{Balanced Accuracy} = \frac{TPR + TNR}{2}
$$

---

### Log Loss

$$
\text{Log Loss}
=
-\frac{1}{n}\sum_{t=1}^{n}
\left[
y_t \log p_t + (1-y_t)\log(1-p_t)
\right]
$$

---

### Brier Score

$$
\text{Brier Score}
=
\frac{1}{n}\sum_{t=1}^{n}(p_t-y_t)^2
$$

---

## Portfolio Evaluation Metrics

### Cumulative return

$$
\prod_{t}(1+R_t)-1
$$

---

### Annualized return

$$
\mu_{ann} = 252 \cdot \bar{R}
$$

---

### Annualized volatility

$$
\sigma_{ann} = \sqrt{252}\cdot \text{Std}(R_t)
$$

---

### Sharpe ratio

$$
\text{Sharpe} = \frac{\mu_{ann}}{\sigma_{ann}}
$$

---

### Maximum drawdown

Let $W_t$ be cumulative wealth:

$$
W_t = \prod_{s\le t}(1+R_s)
$$

Then drawdown is:

$$
DD_t = \frac{W_t}{\max_{s\le t}W_s} - 1
$$

and maximum drawdown is:

$$
MDD = \min_t DD_t
$$

---

### Calmar ratio

$$
\text{Calmar} = \frac{\mu_{ann}}{|MDD|}
$$

---

## Transaction Costs and Turnover

Signal turnover is defined as:

$$
\text{Turnover}_t = \frac{|\hat{d}_t - \hat{d}_{t-1}|}{2}
$$

Net return is:

$$
R_t^{net} = R_t^{gross} - c \cdot \text{Turnover}_t
$$

where $c$ is the assumed transaction cost.  
In this experiment the code uses **5 basis points**.

---

## Benchmarks

### Always long NVDA, short TSLA

$$
R_t^{B1} = \frac{1}{2}(r_{\text{NVDA},t} - r_{\text{TSLA},t})
$$

---

### Always long TSLA, short NVDA

$$
R_t^{B2} = -R_t^{B1}
$$

---

### 21-day momentum benchmark

The **21-day momentum benchmark** uses the relative 21-day momentum between NVDA and TSLA to determine the trading direction.

First define each stock’s 21-day momentum as:

$$
\mathrm{mom21}_{i,t} = \frac{P_{i,t}}{P_{i,t-21}} - 1,
\qquad i \in \{\mathrm{NVDA}, \mathrm{TSLA}\}
$$

Then define the momentum difference:

$$
\mathrm{mom21\_diff}_t
=
\mathrm{mom21}_{\mathrm{NVDA},t}
-
\mathrm{mom21}_{\mathrm{TSLA},t}
$$

The benchmark trading signal is:

$$
d_t^{\mathrm{mom21}} =
\begin{cases}
+1, & \text{if } \mathrm{mom21\_diff}_t \ge 0 \\
-1, & \text{if } \mathrm{mom21\_diff}_t < 0
\end{cases}
$$

So:

- $d_t^{\mathrm{mom21}} = +1$ means **long NVDA and short TSLA**
- $d_t^{\mathrm{mom21}} = -1$ means **long TSLA and short NVDA**
---

### Random Walk

The random-walk benchmark here means:

$$
\hat{d}_t^{R W}=\operatorname{sign}\left(s_t\right)
$$

where

$$
s_t=r_{\mathrm{NVDA}, t}-r_{\mathrm{TSLA}, t}
$$


So for trading on $t+1$ :
- if today NVDA beat TSLA, then tomorrow go long NVDA / short TSLA
- if today TSLA beat NVDA, then tomorrow go long TSLA / short NVDA

---